In [1]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

# CONFIG

In [2]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5*2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [6]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [7]:
unlabelled_path='/kaggle/input/jigsaw-2m-reddit-unlabelled/reddit-removal-log.csv'
unlabelled_df = pd.read_csv(unlabelled_path)

In [8]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [9]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]


# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

Before:(10185, 2)
After: (1875, 2)


,text,label,rule,body,rule_id
0,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...,0
2,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
3,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
4,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...,0


In [10]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [11]:
# print(df.head())
# print(df.columns.tolist())

In [12]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [13]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [14]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [15]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    # print(rule_aucs,'Rule_AUC')
    return avg_auc_per_rule, val_loss, preds

In [17]:
from transformers import get_linear_schedule_with_warmup

In [22]:
def get_subreddit_proportions():
    """Get subreddit proportions for each rule from test data"""
    prop_df = pd.concat((pd.read_csv(test_path),pd.read_csv(train_path)))
    
    proportions = {}
    for rule in prop_df["rule"].unique():
        rule_data = prop_df[prop_df["rule"] == rule]
        subreddit_counts = rule_data["subreddit"].value_counts()
        subreddit_props = subreddit_counts / subreddit_counts.sum()
        proportions[rule] = subreddit_props.to_dict()
    
    return proportions, len(prop_df)
    
def sample_unlabelled_data(multiplier=5):
    """Sample unlabelled data maintaining test data proportions"""
    proportions, test_size = get_subreddit_proportions()
    
    total_sample_size = test_size * multiplier
    sampled_data = []
    
    # Calculate samples per rule based on test data rule distribution
    rule_counts = augmented_df["rule"].value_counts()
    rule_proportions = rule_counts / rule_counts.sum()

    unlabelled_df.drop(unlabelled_df[(unlabelled_df['body'].str.len() > 2000) | (unlabelled_df['body'].str.split().str.len() < 2)].index, inplace=True)    #filter out body present in test,examples
    # excluded_values = augmented_df['body'].values# + train_df[['body', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']].values.ravel()
    # unlabelled_df = unlabelled_df.query('body not in @excluded_values')
    
    for rule, rule_prop in rule_proportions.items():
        rule_sample_size = int(total_sample_size * rule_prop)
        subreddit_props = proportions[rule]
        
        rule_samples = []
        for subreddit, sub_prop in subreddit_props.items():
            subreddit_sample_size = int(rule_sample_size * sub_prop)
            
            # Sample from unlabelled data for this subreddit
            subreddit_data = unlabelled_df[unlabelled_df["subreddit"].str.lower().str.strip() == subreddit.lower().strip()]
            if len(subreddit_data) ==0 or subreddit_sample_size==0: 
                # Fallback: sample from any data for this rule
                continue
                # subreddit_data = unlabelled_df[unlabelled_df["rule"] == rule]

            if len(subreddit_data) >= subreddit_sample_size:
                sampled = subreddit_data.sample(n=subreddit_sample_size, random_state=42)
            else:
                # If not enough data, sample with replacement
                sampled = subreddit_data.sample(n=subreddit_sample_size, replace=True, random_state=42)
            
            sampled = sampled.copy()
            sampled["rule"] = rule
            rule_samples.append(sampled)
        
        if rule_samples:
            sampled_data.append(pd.concat(rule_samples, axis=0))
    
    # Combine all samples and shuffle
    final_sample = pd.concat(sampled_data, axis=0).reset_index(drop=True)
    final_sample = final_sample.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return final_sample
mlm_data= sample_unlabelled_data(3)
mlm_data.head()

,body,subreddit,rule
0,"Correct. The whole sentence is ""Fenwick says t...",news,No legal advice: Do not offer or request legal...
1,Fall down and injure yourself in front of a bu...,personalfinance,No legal advice: Do not offer or request legal...
2,Prostitution is pretty lucrative.,personalfinance,No legal advice: Do not offer or request legal...
3,The liberals are organizing en masse to make s...,The_Donald,No legal advice: Do not offer or request legal...
4,LOL VIDEO 18+ ONLY \nhttps://www.youtube.com/w...,movies,"No Advertising: Spam, referral links, unsolici..."


In [23]:
mlm_data["text"] = mlm_data["rule"] + " [SEP] " + mlm_data["body"]

In [24]:
import math

In [25]:
mlm_texts = mlm_data.text.tolist() #prepare_prepare_mlm_data(df_train, df_test)(df, df_test_full)

from sklearn.model_selection import train_test_split

mlm_train_texts, mlm_val_texts = train_test_split(
    mlm_texts, test_size=0.1, random_state=SEED
)

In [ ]:
# Prepare data for pseudo labeling
pseudo_data = mlm_data.copy()
pseudo_data["text"] = pseudo_data["rule"] + " [SEP] " + pseudo_data["body"]
pseudo_data["rule_id"] = pseudo_data["rule"].str.lower().map(rule_map)

In [27]:
import gc;gc.collect()
torch.cuda.memory.empty_cache()

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # STAGE 1: Train models for pseudo labeling
    print("=== STAGE 1: Training models for pseudo labeling ===")
    
    stage1_models = []
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print(f'--------- FOLD {fold} (Stage 1) --------')
        torch.cuda.empty_cache()
        gc.collect()
        
        val_ds = JigsawDataset(
            augmented_df.iloc[val_idx]['text'].tolist(), 
            augmented_df.iloc[val_idx]['label'].tolist(), 
            augmented_df.iloc[val_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            augmented_df.iloc[tr_idx]['text'].tolist(), 
            augmented_df.iloc[tr_idx]['label'].tolist(), 
            augmented_df.iloc[tr_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, eps=1e-6, weight_decay=3e-2)
        total_steps = EPOCHS * len(train_loader)
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
    
        best_auc = 0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss, val_preds = validate(model, val_loader)
            
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_stage1.bin")
        
        stage1_models.append(f"model_fold{fold}_stage1.bin")
        del model, optimizer, scheduler, train_ds, val_ds, train_loader, val_loader

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Generate pseudo labels using Stage 1 models
    print("=== Generating pseudo labels ===")
    
    pseudo_ds = JigsawDataset(
        pseudo_data['text'].tolist(), 
        [0] * len(pseudo_data),  # dummy labels
        pseudo_data['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )
    pseudo_loader = DataLoader(pseudo_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # Collect predictions from all Stage 1 models
    all_pseudo_preds = []
    
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        model.load_state_dict(torch.load(f"model_fold{fold}_stage1.bin", map_location=DEVICE))
        model.eval()
        
        fold_preds = []
        with torch.no_grad():
            for batch in tqdm(pseudo_loader, desc=f"Pseudo labeling fold {fold}"):
                input_ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                logits = model(input_ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        
        all_pseudo_preds.append(fold_preds)
        del model
        torch.cuda.empty_cache()
    
    # Average predictions across folds
    pseudo_predictions = np.mean(all_pseudo_preds, axis=0)
    
    # Apply high-confidence filtering: >0.9 or <0.1
    high_conf_mask = (pseudo_predictions > 0.9) | (pseudo_predictions < 0.1)
    
    # Convert high-confidence predictions to hard labels
    pseudo_labels = np.where(pseudo_predictions > 0.9, 1.0, 0.0)
    
    # Filter data to keep only high-confidence samples
    filtered_pseudo_data = pseudo_data[high_conf_mask].copy()
    filtered_pseudo_labels = pseudo_labels[high_conf_mask]
    
    # Add filtered pseudo labels to pseudo_data
    filtered_pseudo_data['label'] = filtered_pseudo_labels
    
    # Statistics
    total_samples = len(pseudo_data)
    high_conf_samples = len(filtered_pseudo_data)
    positive_pseudo = (filtered_pseudo_labels == 1.0).sum()
    negative_pseudo = (filtered_pseudo_labels == 0.0).sum()
    
    print(f"Pseudo-labeling Statistics:")
    print(f"  Total unlabeled samples: {total_samples:,}")
    print(f"  High confidence samples (>0.9 or <0.1): {high_conf_samples:,} ({high_conf_samples/total_samples*100:.1f}%)")
    print(f"  Positive pseudo-labels: {positive_pseudo:,} ({positive_pseudo/high_conf_samples*100:.1f}%)")
    print(f"  Negative pseudo-labels: {negative_pseudo:,} ({negative_pseudo/high_conf_samples*100:.1f}%)")
    
    # Combine original augmented data with high-confidence pseudo labeled data
    final_train_data = pd.concat([augmented_df, filtered_pseudo_data[['text', 'label', 'rule_id']]], ignore_index=True)
    print(f"Final training data size: {len(final_train_data):,} (original: {len(augmented_df):,}, pseudo: {len(filtered_pseudo_data):,})")
    print(f"Data increase: {len(final_train_data)/len(augmented_df):.1f}x")

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # STAGE 2: Train final models on augmented data (original + pseudo labeled)
    print("=== STAGE 2: Training final models on augmented data ===")
    
    stage2_models = []
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    
    for fold, (tr_idx, val_idx) in enumerate(folds.split(final_train_data, final_train_data["rule_id"])):
        print(f'--------- FOLD {fold} (Stage 2) --------')
        torch.cuda.empty_cache()
        gc.collect()
        
        val_ds = JigsawDataset(
            final_train_data.iloc[val_idx]['text'].tolist(), 
            final_train_data.iloc[val_idx]['label'].tolist(), 
            final_train_data.iloc[val_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            final_train_data.iloc[tr_idx]['text'].tolist(), 
            final_train_data.iloc[tr_idx]['label'].tolist(), 
            final_train_data.iloc[tr_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize fresh model from scratch
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, eps=1e-6, weight_decay=3e-2)
        total_steps = EPOCHS * len(train_loader)
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
    
        best_auc = 0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss, val_preds = validate(model, val_loader)
            
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_final.bin")
        
        stage2_models.append(f"model_fold{fold}_final.bin")
        print(f"Best AUC for fold {fold}: {best_auc:.4f}")
        del model, optimizer, scheduler, train_ds, val_ds, train_loader, val_loader

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Generate final predictions using Stage 2 models
    print("=== Generating final predictions ===")
    
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        model.load_state_dict(torch.load(f"model_fold{fold}_final.bin", map_location=DEVICE))
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Predicting fold {fold}"):
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        
        test_preds.append(fold_preds)
        del model
        torch.cuda.empty_cache()

In [31]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv